In [1]:
import pandas as pd

# Read each table
ccat = pd.read_csv('C:/Users/hyjfo/OneDrive/MDACC/P1. CRAD in mucinous CRC/Waddington_landscape/For_WI_ACI/CCAT.csv')
phate = pd.read_csv('C:/Users/hyjfo/OneDrive/MDACC/P1. CRAD in mucinous CRC/Waddington_landscape/For_WI_ACI/PHATE.csv')
sample_cluster = pd.read_csv('C:/Users/hyjfo/OneDrive/MDACC/P1. CRAD in mucinous CRC/Waddington_landscape/For_WI_ACI/Sample_Cluster.csv')
velocity_length = pd.read_csv('C:/Users/hyjfo/OneDrive/MDACC/P1. CRAD in mucinous CRC/Waddington_landscape/For_WI_ACI/velocity_length.csv')

# Merge all tables on the 'Cell' column
merged_df = phate[['Cell', 'PHATE1', 'PHATE2']].merge(
    sample_cluster[['Cell', 'batch', 'leiden']],
    on='Cell',
    how='outer'
).merge(
    ccat[['Cell', 'CCAT']],
    on='Cell',
    how='outer'
).merge(
    velocity_length[['Cell', 'velocity_length']],
    on='Cell',
    how='outer'
)

# Save the merged table
output_path = 'C:/Users/hyjfo/OneDrive/MDACC/P1. CRAD in mucinous CRC/Waddington_landscape/For_WI_ACI/md_scRNAseq.csv'
merged_df.to_csv(output_path, index=False)


In [2]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import graphtools
from scipy.interpolate import Rbf
from sklearn.preprocessing import MinMaxScaler

#This grid sets both range of phate dims and the number of "ticks" -> controls "resolution"
grid_x, grid_y = np.mgrid[-0.035:0.045:0.0024, -0.025:0.045:0.0024]

In [3]:
import plotly.io as pio

# This ensures Plotly output works in multiple places:
# plotly_mimetype: VS Code notebook UI
# notebook: "Jupyter: Export to HTML" command in VS Code
# See https://plotly.com/python/renderers/#multiple-renderers
pio.renderers.default = "plotly_mimetype+notebook"

In [4]:
#Functions

def compute_distdeg(input_df, split_by="leiden", 
    col_list=["PHATE1","PHATE2"], 
    knn=30, distance_metric = "manhattan", scale = True, inv_degree=True,
    plot=False, rmv_outliers=True):
    #Function that computes distances and degrees of cells within a cluster
    #Results will get scaled within the cluster level so it handles them as independent entities
    output_df = pd.DataFrame(columns=col_list+["dist","deg"])
    for i in input_df[split_by].unique():
        print(f"Processing cluster: {i}")
        df_dist = input_df.loc[input_df[split_by]==i,col_list]
        #Knn param is key and must be tunned according to the size of the cluster
        G = graphtools.Graph(df_dist.values, knn=knn, decay=None, distance=distance_metric)
        # Return distances by using shortest_path
        np_dist = G.shortest_path(distance="data")
        #Compute median distance to all cells for al cells -> min should be a centroid
        df_dist["dist"] = np.median(np_dist, axis=1)
        df_dist["deg"] = G.to_pygsp().d
        if rmv_outliers:
            #Replace distant outliers with median value -> DONE to smoothen landscape at the very edges of a cluster
            df_dist["dist"] = np.where(
                                df_dist["dist"]>np.percentile(df_dist["dist"],99), 
                                df_dist["dist"].median(), 
                                df_dist["dist"])
        if scale:
            df_dist["dist_scaled"] = MinMaxScaler().fit_transform(X = df_dist[["dist"]])
            df_dist["inv_deg_scaled"] = MinMaxScaler().fit_transform(X = 1/df_dist[["deg"]])
        if plot:
            #Plot cluster cells on phate and colour by distance from centroid
            plt.scatter(df_dist["PHATE1"],df_dist["PHATE2"],c=df_dist["dist"])
            plt.show()
            plt.scatter(df_dist["PHATE1"],df_dist["PHATE2"],c=df_dist["deg"])
            plt.show()

        output_df = pd.concat([output_df, df_dist])

    return output_df


In [5]:
cell_data = pd.read_csv("C:/Users/hyjfo/OneDrive/MDACC/P1. CRAD in mucinous CRC/Waddington_landscape/For_WI_ACI/md_scRNAseq.csv")
cell_data

,Cell,PHATE1,PHATE2,batch,leiden,CCAT,velocity_length
0,AAACCCAAGAGGCTGT-1-WI,0.002597,-0.008322,WI,Progenitors_4,0.371739,13.80
1,AAACCCAAGCCATGCC-1-WI,-0.010205,0.002499,WI,Stem,0.164988,15.15
2,AAACCCAAGTGCTCAT-1-WI,-0.003618,-0.015389,WI,Progenitors_1,0.273011,16.00
3,AAACCCACAACAGATA-1-WI,-0.008073,0.002430,WI,Colonocytes_1,0.303201,22.42
4,AAACCCACACACCTAA-1-WI,-0.007192,0.002376,WI,Progenitors_3,0.366847,26.52
...,...,...,...,...,...,...,...
28643,TTTGTTGTCAAGTAAG-1-ACI,-0.008493,0.002151,ACI,Progenitors_4,0.275045,22.12
28644,TTTGTTGTCAGCGGAA-1-ACI,-0.008377,0.000513,ACI,Colonocytes_6,0.302192,31.82
28645,TTTGTTGTCGACGCTG-1-ACI,-0.010937,0.004791,ACI,Colonocytes_1,0.099806,45.71
28646,TTTGTTGTCTACGCAA-1-ACI,-0.008353,0.001646,ACI,Colonocytes_2,0.294446,32.19


In [6]:
cell_data_1 = cell_data[cell_data['leiden'].isin(['Stem', 'Progenitors_1', 'Progenitors_2', 'Progenitors_3', 'Progenitors_4',
                                                 'Colonocytes_1', 'Colonocytes_2', 'Colonocytes_3', 'Colonocytes_5', 'Colonocytes_6', 'Colonocytes_7',
                                                 'Goblet_1', 'Goblet_2', 'Enteroendocrine', 'Tuft'])]
cell_data_1

,Cell,PHATE1,PHATE2,batch,leiden,CCAT,velocity_length
0,AAACCCAAGAGGCTGT-1-WI,0.002597,-0.008322,WI,Progenitors_4,0.371739,13.80
1,AAACCCAAGCCATGCC-1-WI,-0.010205,0.002499,WI,Stem,0.164988,15.15
2,AAACCCAAGTGCTCAT-1-WI,-0.003618,-0.015389,WI,Progenitors_1,0.273011,16.00
3,AAACCCACAACAGATA-1-WI,-0.008073,0.002430,WI,Colonocytes_1,0.303201,22.42
4,AAACCCACACACCTAA-1-WI,-0.007192,0.002376,WI,Progenitors_3,0.366847,26.52
...,...,...,...,...,...,...,...
28643,TTTGTTGTCAAGTAAG-1-ACI,-0.008493,0.002151,ACI,Progenitors_4,0.275045,22.12
28644,TTTGTTGTCAGCGGAA-1-ACI,-0.008377,0.000513,ACI,Colonocytes_6,0.302192,31.82
28645,TTTGTTGTCGACGCTG-1-ACI,-0.010937,0.004791,ACI,Colonocytes_1,0.099806,45.71
28646,TTTGTTGTCTACGCAA-1-ACI,-0.008353,0.001646,ACI,Colonocytes_2,0.294446,32.19


In [7]:
empty_df_1 = compute_distdeg(cell_data_1, knn=30, plot=False)
# knn should be tunned for every single cell cluster according to their size. If knn is too large or too small, ValueError: csgraph must be lil, csr, or csc format.
# The best knn should be optimized for every single cell cluster to find the best distance to descible the real cell transitions. Here, I just use 30 for lazy.

Processing cluster: Progenitors_4
Processing cluster: Stem
Processing cluster: Progenitors_1
Processing cluster: Colonocytes_1
Processing cluster: Progenitors_3
Processing cluster: Colonocytes_3
Processing cluster: Colonocytes_2
Processing cluster: Colonocytes_6
Processing cluster: Goblet_2
Processing cluster: Colonocytes_5
Processing cluster: Goblet_1
Processing cluster: Progenitors_2
Processing cluster: Enteroendocrine
Processing cluster: Tuft
Processing cluster: Colonocytes_7


In [8]:
# Add distances and degrees to main dataframe
cell_data_1 = pd.merge(cell_data_1, empty_df_1.sort_index(), how="left",on=["PHATE1","PHATE2"])

# Scale velocity length for all cells
cell_data_1["inv_velo_scaled"] = MinMaxScaler().fit_transform(X = 1/cell_data_1[["velocity_length"]])

# Remove duplicated Cells
cell_data_1_unique = cell_data_1.drop_duplicates(subset='Cell', keep='first')
cell_data_1_unique

,Cell,PHATE1,PHATE2,batch,leiden,CCAT,velocity_length,dist,deg,dist_scaled,inv_deg_scaled,inv_velo_scaled
0,AAACCCAAGAGGCTGT-1-WI,0.002597,-0.008322,WI,Progenitors_4,0.371739,13.80,0.015419,35,0.266268,0.685714,0.252229
1,AAACCCAAGCCATGCC-1-WI,-0.010205,0.002499,WI,Stem,0.164988,15.15,0.001698,34,0.001775,0.721925,0.225886
2,AAACCCAAGTGCTCAT-1-WI,-0.003618,-0.015389,WI,Progenitors_1,0.273011,16.00,0.018561,35,0.240412,0.642857,0.211580
3,AAACCCACAACAGATA-1-WI,-0.008073,0.002430,WI,Colonocytes_1,0.303201,22.42,0.004771,37,0.061742,0.615098,0.138566
4,AAACCCACACACCTAA-1-WI,-0.007192,0.002376,WI,Progenitors_3,0.366847,26.52,0.001626,41,0.010988,0.396341,0.110434
...,...,...,...,...,...,...,...,...,...,...,...,...
28727,TTTGTTGTCAAGTAAG-1-ACI,-0.008493,0.002151,ACI,Progenitors_4,0.275045,22.12,0.000738,31,0.000793,0.929032,0.141034
28728,TTTGTTGTCAGCGGAA-1-ACI,-0.008377,0.000513,ACI,Colonocytes_6,0.302192,31.82,0.005758,35,0.008142,0.698413,0.084811
28729,TTTGTTGTCGACGCTG-1-ACI,-0.010937,0.004791,ACI,Colonocytes_1,0.099806,45.71,0.002451,39,0.004068,0.530504,0.045851
28730,TTTGTTGTCTACGCAA-1-ACI,-0.008353,0.001646,ACI,Colonocytes_2,0.294446,32.19,0.003042,41,0.006704,0.409756,0.083337


In [9]:
cell_data_2 = cell_data[cell_data['leiden'].isin(['Colonocytes_4'])]
cell_data_2

,Cell,PHATE1,PHATE2,batch,leiden,CCAT,velocity_length
254,AACGAAAGTGAGTTTC-1-WI,-0.013592,-0.002771,WI,Colonocytes_4,0.214365,12.36
1144,ACTGCAATCCATGAGT-1-WI,0.000905,-0.001735,WI,Colonocytes_4,0.381258,11.31
2136,ATACTTCTCTTTGCAT-1-WI,0.029939,-0.009911,WI,Colonocytes_4,0.355432,15.55
4359,CGAGTGCAGCTGTCCG-1-WI,-0.008447,0.001763,WI,Colonocytes_4,0.309276,15.70
6560,GCCCGAATCACCATGA-1-WI,-0.008456,0.000745,WI,Colonocytes_4,0.308335,22.06
8343,TAAGCACGTCTGTGGC-1-WI,-0.007663,-0.002344,WI,Colonocytes_4,0.303854,25.72
9959,TGACCCTTCTCGGTCT-1-WI,-0.008684,0.000385,WI,Colonocytes_4,0.278535,23.90
12566,ACAAGCTGTGAATTGA-1-ACI,-0.008991,0.001348,ACI,Colonocytes_4,0.301433,23.62
13589,AGAAGCGTCAATCTCT-1-ACI,-0.010408,0.002001,ACI,Colonocytes_4,0.175739,35.54
13669,AGACCATCACGAAGAC-1-ACI,-0.008378,-0.002194,ACI,Colonocytes_4,0.244320,39.67


In [10]:
empty_df_2 = compute_distdeg(cell_data_2, knn=2, plot=False)

Processing cluster: Colonocytes_4


In [11]:
# Add distances and degrees to main dataframe
cell_data_2 = pd.merge(cell_data_2, empty_df_2.sort_index(), how="left",on=["PHATE1","PHATE2"])

# Scale velocity length for all cells
cell_data_2["inv_velo_scaled"] = MinMaxScaler().fit_transform(X = 1/cell_data_2[["velocity_length"]])

# Remove duplicated Cells
cell_data_2_unique = cell_data_2.drop_duplicates(subset='Cell', keep='first')
cell_data_2_unique

,Cell,PHATE1,PHATE2,batch,leiden,CCAT,velocity_length,dist,deg,dist_scaled,inv_deg_scaled,inv_velo_scaled
0,AACGAAAGTGAGTTTC-1-WI,-0.013592,-0.002771,WI,Colonocytes_4,0.214365,12.36,0.007979,2,0.340340,1.00,0.886411
1,ACTGCAATCCATGAGT-1-WI,0.000905,-0.001735,WI,Colonocytes_4,0.381258,11.31,0.017724,3,0.863038,0.50,1.000000
2,ATACTTCTCTTTGCAT-1-WI,0.029939,-0.009911,WI,Colonocytes_4,0.355432,15.55,0.002438,2,0.043159,1.00,0.635412
3,CGAGTGCAGCTGTCCG-1-WI,-0.008447,0.001763,WI,Colonocytes_4,0.309276,15.70,0.001792,3,0.008512,0.50,0.626121
4,GCCCGAATCACCATGA-1-WI,-0.008456,0.000745,WI,Colonocytes_4,0.308335,22.06,0.001921,3,0.015429,0.50,0.348417
5,TAAGCACGTCTGTGGC-1-WI,-0.007663,-0.002344,WI,Colonocytes_4,0.303854,25.72,0.005219,3,0.192338,0.50,0.250866
6,TGACCCTTCTCGGTCT-1-WI,-0.008684,0.000385,WI,Colonocytes_4,0.278535,23.90,0.002280,3,0.034665,0.50,0.295640
7,ACAAGCTGTGAATTGA-1-ACI,-0.008991,0.001348,ACI,Colonocytes_4,0.301433,23.62,0.001825,2,0.010266,1.00,0.303141
8,AGAAGCGTCAATCTCT-1-ACI,-0.010408,0.002001,ACI,Colonocytes_4,0.175739,35.54,0.003090,3,0.078112,0.50,0.088403
9,AGACCATCACGAAGAC-1-ACI,-0.008378,-0.002194,ACI,Colonocytes_4,0.244320,39.67,0.004510,2,0.154294,1.00,0.044104


In [12]:
data = pd.concat([cell_data_1_unique, cell_data_2_unique], axis=0)
data

,Cell,PHATE1,PHATE2,batch,leiden,CCAT,velocity_length,dist,deg,dist_scaled,inv_deg_scaled,inv_velo_scaled
0,AAACCCAAGAGGCTGT-1-WI,0.002597,-0.008322,WI,Progenitors_4,0.371739,13.80,0.015419,35,0.266268,0.685714,0.252229
1,AAACCCAAGCCATGCC-1-WI,-0.010205,0.002499,WI,Stem,0.164988,15.15,0.001698,34,0.001775,0.721925,0.225886
2,AAACCCAAGTGCTCAT-1-WI,-0.003618,-0.015389,WI,Progenitors_1,0.273011,16.00,0.018561,35,0.240412,0.642857,0.211580
3,AAACCCACAACAGATA-1-WI,-0.008073,0.002430,WI,Colonocytes_1,0.303201,22.42,0.004771,37,0.061742,0.615098,0.138566
4,AAACCCACACACCTAA-1-WI,-0.007192,0.002376,WI,Progenitors_3,0.366847,26.52,0.001626,41,0.010988,0.396341,0.110434
...,...,...,...,...,...,...,...,...,...,...,...,...
25,GTTAGTGTCTCAACCC-1-ACI,-0.008677,0.001787,ACI,Colonocytes_4,0.313179,25.03,0.001738,3,0.005582,0.500000,0.267074
26,TCTAACTCAGAGACTG-1-ACI,-0.008596,0.002188,ACI,Colonocytes_4,0.259733,30.69,0.001883,5,0.013382,0.100000,0.155648
27,TCTGGCTAGGGTAGCT-1-ACI,-0.014611,0.001120,ACI,Colonocytes_4,0.141496,27.94,0.007504,2,0.314864,1.000000,0.204147
28,TTACCATTCGCTAATG-1-ACI,-0.004772,-0.006674,ACI,Colonocytes_4,0.305901,33.56,0.010147,2,0.456610,1.000000,0.113508


In [19]:
data["median_CCAT"] = data.groupby(by="leiden")[["CCAT"]].transform("median")
data["median_inv_velo"]= data.groupby(by="leiden")[["inv_velo_scaled"]].transform("median")
data["VR"] = data.apply(lambda x: 0.9*x["median_CCAT"]+0.1*(x["median_inv_velo"]*x["dist_scaled"]), axis=1)
data

,Cell,PHATE1,PHATE2,batch,leiden,CCAT,velocity_length,dist,deg,dist_scaled,inv_deg_scaled,inv_velo_scaled,median_CCAT,median_inv_velo,VR
0,AAACCCAAGAGGCTGT-1-WI,0.002597,-0.008322,WI,Progenitors_4,0.371739,13.80,0.015419,35,0.266268,0.685714,0.252229,0.317439,0.193584,0.290850
1,AAACCCAAGCCATGCC-1-WI,-0.010205,0.002499,WI,Stem,0.164988,15.15,0.001698,34,0.001775,0.721925,0.225886,0.162668,0.121970,0.146423
2,AAACCCAAGTGCTCAT-1-WI,-0.003618,-0.015389,WI,Progenitors_1,0.273011,16.00,0.018561,35,0.240412,0.642857,0.211580,0.354400,0.193103,0.323602
3,AAACCCACAACAGATA-1-WI,-0.008073,0.002430,WI,Colonocytes_1,0.303201,22.42,0.004771,37,0.061742,0.615098,0.138566,0.089518,0.109140,0.081240
4,AAACCCACACACCTAA-1-WI,-0.007192,0.002376,WI,Progenitors_3,0.366847,26.52,0.001626,41,0.010988,0.396341,0.110434,0.327676,0.141201,0.295064
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,GTTAGTGTCTCAACCC-1-ACI,-0.008677,0.001787,ACI,Colonocytes_4,0.313179,25.03,0.001738,3,0.005582,0.500000,0.267074,0.287764,0.233350,0.259118
26,TCTAACTCAGAGACTG-1-ACI,-0.008596,0.002188,ACI,Colonocytes_4,0.259733,30.69,0.001883,5,0.013382,0.100000,0.155648,0.287764,0.233350,0.259300
27,TCTGGCTAGGGTAGCT-1-ACI,-0.014611,0.001120,ACI,Colonocytes_4,0.141496,27.94,0.007504,2,0.314864,1.000000,0.204147,0.287764,0.233350,0.266335
28,TTACCATTCGCTAATG-1-ACI,-0.004772,-0.006674,ACI,Colonocytes_4,0.305901,33.56,0.010147,2,0.456610,1.000000,0.113508,0.287764,0.233350,0.269642


In [20]:
data.to_csv("C:/Users/hyjfo/OneDrive/MDACC/P1. CRAD in mucinous CRC/Waddington_landscape/For_WI_ACI/Waddington_axis.csv", index=False)